Categorical plots: https://seaborn.pydata.org/tutorial/categorical.html

V-Measure paper (with definition of F-Measure): https://www.aclweb.org/anthology/D07-1043.pdf

In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="ticks", color_codes=True)

from collections import Counter
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics.cluster import adjusted_mutual_info_score
from weasyprint import HTML

from utils.analysis import align_clusterings_for_sklearn
from utils.io import load_clustering, get_review_pmids
from utils.metrics import pd_score, reg_v_score
from utils.preprocessing import preprocess_clustering, get_clustering_level

### Load Raw Grid Search Output

In [2]:
# Old
RESULTS_JSON_ORIGINAL = '/home/willenjoy/work/pubtrends/local/nature_reviews/grid_search/full_2021_03_03/grid_search_full_2021-03-03_13_22.json'
RESULTS_JSON_COCIT_THRESHOLD = '/home/willenjoy/work/pubtrends/local/nature_reviews/grid_search/min_cocitation=2_2021_03_10/grid_search_min_cocitation=2_2021-03-10_01_37.json'
RESULTS_JSON_NEW_EXPAND = '/home/willenjoy/work/pubtrends/local/nature_reviews/grid_search/new_expand_2021_03_10/grid_search_new_expand_2021-03-10_22_30.json'
# Current
RESULTS_JSON_AFTER_MERGE = '/home/willenjoy/work/pubtrends/local/nature_reviews/grid_search/grid_search_2021_04_11/grid_search_2021-04-11_22_30.json'

In [3]:
results = []

In [4]:
with open(RESULTS_JSON_AFTER_MERGE, 'r') as f:
    results = json.load(f)

### Load Clusterings

In [5]:
ground_truth = {}

for pmid in get_review_pmids():
    clustering = load_clustering(pmid)
    for level in range(1, get_clustering_level(clustering)):
        ground_truth[(pmid, level)] = preprocess_clustering(clustering, max_level=level,
                                                            include_box_sections=False,
                                                            uniqueness_method='unique_only')

### Generate Score Dataframe From Grid Search Results

#### Add Regularized V-Score Column (for Version 1 Output)

In [ ]:
def add_reg_column(score_df, reg):
    reg_column = f'Reg-V-Score'
    score_df[reg_column] = score_df['V-Score'] - reg * score_df['Clusters']
    return score_df, reg_column

In [6]:
PARAM_NAMES_OLD = ['bibcoupling', 'cocitation', 'citation', 'text_citation']
PARAM_NAMES = ['cocitation', 'bibcoupling', 'citation', 'text_citation']

REG_WEIGHT = 0.01

In [ ]:
def get_score_dataframe_v1(results):
    score_data = []
    partitions = {}

    for paper in results:
        pmid = paper['pmid']
        level = paper['level']
        full_id = f'{pmid} ({level})'
        grid_results = paper['results']
        for result in grid_results:
            soi = result['soi']
            data = result['results']
            partition = data['partition']
            score = data['v_measure_score']

            max_soi = max(soi)
            main_param_idx = soi.index(max_soi)
            main_param = PARAM_NAMES[main_param_idx]
            n_clusters = len(set(partition.values()))

            labels_true, labels_pred = align_clusterings_for_sklearn(ground_truth[(pmid, level)], partition)
            
            score_data.append((pmid, level, *soi, 
                               score, data['adjusted_mutual_info_score'], pd_score(labels_true, labels_pred), 
                               n_clusters, main_param, full_id))
            partitions[(pmid, level, *soi)] = partition
            
    score_df = pd.DataFrame(score_data, columns=['PMID', 'Level', *PARAM_NAMES, 
                                                 'V-Score', 'AMI-Score', 'PD-Score',
                                                 'Clusters', 'Largest Parameter', 'PMID (level)'])
    score_df, reg_column = add_reg_column(score_df, reg=REG_WEIGHT)
    return score_df

In [ ]:
def get_score_dataframe_v2(results):
    score_data = []
    partitions = {}

    for paper in results:
        pmid = paper['pmid']
        level_results = paper['results']
        for level, grid_results in level_results.items():
            full_id = f'{pmid} ({level})'
            for result in grid_results:
                soi = result['soi']
                data = result['results']
                partition = data['partition']

                max_soi = max(soi)
                main_param_idx = soi.index(max_soi)
                main_param = PARAM_NAMES_V2[main_param_idx]
                n_clusters = len(set(partition.values()))

                labels_true, labels_pred = align_clusterings_for_sklearn(ground_truth[(pmid, int(level))], 
                                                                         partition)
                scores = [pd_score(labels_true, labels_pred), data['reg_v_score'], 
                          data['v_measure_score'], data['adjusted_mutual_info_score']]

                score_data.append((pmid, level, *soi, *scores, 
                                   n_clusters, main_param, full_id))
                partitions[(pmid, level, *soi)] = partition
            
    score_df = pd.DataFrame(score_data, columns=['PMID', 'Level', *PARAM_NAMES_V2, 
                                                 'PD-Score', 'Reg-V-Score', 'V-Score', 'AMI-Score',
                                                 'Clusters', 'Largest Parameter', 'PMID (level)'])
    return score_df

In [ ]:
SIZE_CATEGORIES = {
    'Small (<10%)': (0, 0.1),
    'Medium (10, 20%)': (0.1, 0.2),
    'Big (20, 40%)': (0.2, 0.4),
    'Large (>40%)': (0.4, 1),
}

METRICS = [
    pd_score,
    reg_v_score,
    adjusted_mutual_info_score
]

def get_score_by_size_dataframe_v1(results):
    score_data = []
    partitions = {}

    for paper in results:
        pmid = paper['pmid']
        level = paper['level']
        full_id = f'{pmid} ({level})'
        grid_results = paper['results']
        for result in grid_results:
            soi = result['soi']
            data = result['results']
            partition = data['partition']
            
            max_soi = max(soi)
            main_param_idx = soi.index(max_soi)
            main_param = PARAM_NAMES[main_param_idx]
            
            gt_partition = ground_truth[(pmid, level)]
            n_papers = len(gt_partition)
            gt_counter = Counter(gt_partition.values())
            for k in gt_counter:
                gt_counter[k] /= n_papers
            for size_label, (min_size, max_size) in SIZE_CATEGORIES.items():
                accepted_clusters = [el[0] for el in
                                     list(filter(lambda el: min_size < el[1] <= max_size, gt_counter.items()))]
                if len(accepted_clusters) == 0:
                    continue
                    
                # Filter ground truth partition
                filtered_gt_partition = {k: v for k, v in gt_partition.items() if v in accepted_clusters}                
                
                # Pick the same PMIDs from PubTrends partition
                filtered_partition = {k: v for k, v in partition.items() if k in filtered_gt_partition}                

                labels_true, labels_pred = align_clusterings_for_sklearn(filtered_gt_partition, 
                                                                         filtered_partition)
                scores = []
                for metric in METRICS:
                    scores.append(metric(labels_true, labels_pred))

                score_data.append((pmid, level, size_label, *soi, *scores, main_param, full_id))
                partitions[(pmid, level, *soi)] = partition

    score_df = pd.DataFrame(score_data, columns=['PMID', 'Level', 'Cluster Size', *PARAM_NAMES, 
                                                 'PD-Score', 'Reg-V-Score', 'AMI-Score',
                                                 'Largest Parameter', 'PMID (level)'])
    return score_df

## Visualizations Within One Grid Search

In [ ]:
score_df = get_score_dataframe_v1(results)

In [ ]:
score_df.head(5)

In [ ]:
score_cols = ['PD-Score', 'AMI-Score', 'Reg-V-Score']

### Print Top Parameter Sets by Mean Value for All Review Papers

In [ ]:
def get_top_parameter_sets(score_df, target_col, n=5):
    return score_df.groupby(PARAM_NAMES_V2)[[target_col, 'Clusters']].mean().sort_values(by=target_col, 
                                                                                         ascending=False).head(n).reset_index()

In [ ]:
def get_top_mean_score(score_df, target_col):
    return score_df.groupby(PARAM_NAMES_V2)[target_col].mean().sort_values(ascending=False).values[0]

In [ ]:
for col in score_cols:
    print(col, get_top_mean_score(score_df, col), '\n')
    print(get_top_parameter_sets(score_df, col), '\n')

#### Top Per Hierarchy Level

In [ ]:
def get_top_parameter_sets_per_level(target_col, n=5):
    return score_df.groupby(['Level'] + PARAM_NAMES_V2)[target_col].mean().reset_index()

In [ ]:
def get_top_mean_score_per_level(target_col, level):
    top_scores_df = score_df.groupby(['Level'] + PARAM_NAMES)[target_col].mean().reset_index()
    top_scores_per_level = top_scores_df[top_scores_df['Level'] == level].sort_values(target_col, ascending=False)
    return top_scores_per_level[target_col].values[0]

In [ ]:
levels = [1, 2, 3]

for col in score_cols:
    print('\n', col, '\n')
    top_parameter_sets = get_top_parameter_sets_per_level(col)
    for level in levels:
        print(f'Level {level}: {get_top_mean_score_per_level(col, level)}')
        print(top_parameter_sets[top_parameter_sets['Level'] == level].sort_values(col, ascending=False).head(3))

### Get Individually Best Clustering Parameters

In [ ]:
def get_best_clustering(score_df, target_col, unique=False):
    cols_to_select = ['PMID', 'Level', *PARAM_NAMES, target_col, 'Clusters']
    idx = score_df.groupby('PMID (level)')[target_col].transform(max) == score_df[target_col]
    if unique:
        return score_df.loc[idx, cols_to_select].drop_duplicates(subset=['PMID', 'Level'])
    return score_df.loc[idx, cols_to_select]

In [ ]:
best_clustering_df = get_best_clustering(score_df, 'Reg-V-Score', unique=True)

In [ ]:
best_clustering_df

### Boxplot of Individually Best Scores

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for score_col, ax in zip(score_cols, axes):
    best_clustering_score_df = get_best_clustering(score_df, score_col, unique=True)
    best_mean_score = get_top_mean_score(score_df, score_col)
    sns.boxplot(ax=ax, x="Level", y=score_col, data=best_clustering_score_df)
    ax.axhline(best_mean_score, 0, 1, label='Best Mean Score', linestyle='--', c='red')
    ax.legend()

### Individual Best Scores vs Variance

In [ ]:
max_vs_std_df = score_df.groupby(['PMID', 'Level'])[score_cols].agg(['max', 'std'])

In [ ]:
pd.plotting.scatter_matrix(max_vs_std_df, figsize=(18, 18));

### Score Distributions for Different Reseach Fields

In [ ]:
REVIEW_FIELD_MAPPING = {
    'cancer': [26667849, 26678314, 27834397, 27834398, 29147025, 
               29170536, 30578414, 30842595, 31686003, 31806885],
    'immunology': [26688349, 26688350, 27890914, 27916977, 28853444, 
                   28920587, 30644449, 30679807, 31836872, 32005979],
    'molecular cell biology': [26580716, 26580717, 27677859, 27677860, 28792006, 
                               28852220, 30108335, 30390028, 31937935, 32020081],
    'neuroscience': [26656254, 26675821, 27904142, 28003656, 29213134, 
                     29321682, 30459365, 30467385, 32042144, 32699292]
}

In [ ]:
reversed_mapping = {}
for field, pmids in REVIEW_FIELD_MAPPING.items():
    for pmid in pmids:
        reversed_mapping[str(pmid)] = field

In [ ]:
field_df = pd.DataFrame(reversed_mapping.items(), columns=['PMID', 'field'])

In [ ]:
score_df = pd.merge(score_df, field_df, on='PMID')

In [ ]:
best_score_df = score_df.groupby(['PMID', 'Level', 'field'])[score_cols].agg('max').reset_index()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.swarmplot(ax=ax, x='Reg-V-Score', y='field', data=best_score_df)
ax.text(x=0.5, y=0.9, s="Saturday\nMean", horizontalalignment='center')

In [ ]:
best_score_df = best_score_df.sort_values(by=['field', 'Reg-V-Score'], ascending=[True, False])

In [ ]:
print_dataframe_to_pdf(best_score_df, f'best_scores_per_field.pdf')

### Best Mean Scores for Different Levels

In [ ]:
fig, ax = plt.subplots()
for score_col in score_cols:
    mean_scores_per_level = [get_top_mean_score_per_level(score_col, level) for level in levels]
    str_levels = [str(l) for l in levels]
    ax.plot(str_levels, mean_scores_per_level, 'o-', label=score_col)
ax.set_xlabel('Level')
ax.set_ylabel('Score')
ax.legend()

### Print Individually Best Parameters to PDF

In [ ]:
def print_dataframe_to_pdf(df, filename):
    table = df.to_html(classes='mystyle', index=False)

    html_string = f'''
    <html>
      <head><title>HTML Pandas Dataframe with CSS</title></head>
      <link rel="stylesheet" type="text/css" href="df_style.css"/>
      <body>
        {table}
      </body>
    </html>
    '''

    HTML(string=html_string).write_pdf(filename, stylesheets=["df_style.css"])

In [ ]:
print_dataframe_to_pdf(best_clustering_df, f'best_clustering_stats_V_score-{reg}*Clusters.pdf')

### Plot Contingency Matrices for the Individually Best Clusterings

In [ ]:
def plot_contingency_matrix(cm, title_str):
    fig = plt.figure(figsize = (10,7))
    sns.heatmap(cm, annot=True)
    plt.title(title_str)
    plt.xlabel('PubTrends')
    plt.ylabel('Nature Reviews')
    return fig

In [ ]:
with PdfPages(f'best_clusterings_V_score-{reg}*Clusters.pdf') as pdf:
    for index, row in best_clustering_df.iterrows():
        p_key = (row['PMID'], row['Level'], 
                 row['bibcoupling'], row['cocitation'], row['citation'], row['text_citation'])
        gt_key = (row['PMID'], row['Level'])
        partition = partitions[p_key]
        gt = ground_truth[gt_key]
        labels_true, labels_pred = align_clusterings_for_sklearn(partition, gt)
        contingency = contingency_matrix(labels_true, labels_pred)
        
        title_str = f"{row['PMID']} - LEVEL {row['Level']}\nReg-Score: {row[reg_column]}"
        fig = plot_contingency_matrix(contingency, title_str)
        pdf.savefig(fig)
        plt.close()
        
        print('.', end='')

### Plot All Grid Search Results for All Review Papers

In [ ]:
def plot_catplot(score_df, target_col, filename):
    g = sns.catplot(x=target_col, y="PMID (level)", hue="Largest Parameter", 
                    kind="strip", data=score_df, height=40, aspect=0.5)
    g.savefig(filename)

In [ ]:
plot_catplot(score_df, reg_column, f'full_grid_search_V_score-{reg}*Clusters.png')

### Plot Mean Score Changes for Different Sets of Parameters

In [ ]:
import matplotlib.pyplot as plt

def plot_param_grid(param_grid_array, ax, vmin, vmax, param_range, row_label, col_label, title):
    im = ax.imshow(param_grid_array, vmin=vmin, vmax=vmax, cmap="viridis")
    
    # We want to show all ticks...
    n_params = len(param_range)
    ax.set_xticks(np.arange(n_params))
    ax.set_yticks(np.arange(n_params))
    # ... and label them with the respective list entries
    ax.set_xticklabels(param_range)
    ax.set_yticklabels(param_range)
    ax.set_xlabel(col_label)
    ax.set_ylabel(row_label)
    ax.set_title(title, fontweight='bold')

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
             rotation_mode="anchor")

    for i in range(n_params):
        for j in range(n_params):
            display_value = round(param_grid_array[i, j], 2)
            text = ax.text(j, i, display_value,
                           ha="center", va="center", color="w")

In [ ]:
# TARGET_COLUMN = 'PD-Score'
TARGET_COLUMN = 'Reg-V-Score'

In [ ]:
mean_score_df = score_df.groupby(PARAM_NAMES_V2)[TARGET_COLUMN].mean().reset_index()

In [ ]:
with_cocitation = mean_score_df.cocitation == 1
print(mean_score_df[with_cocitation].groupby(['bibcoupling'])[TARGET_COLUMN].mean())

In [ ]:
param_range = [0, 0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8, 16]
param_idx = {param: i for i, param in enumerate(param_range)}

n_params = len(param_range)
param_grid_array = np.zeros((n_params, n_params))  

In [ ]:
# Min should be at least 0, negative values are the least important, but mess up the color scheme
vmin = max(mean_score_df[TARGET_COLUMN].min(), 0)
vmax = mean_score_df[TARGET_COLUMN].max()
print(vmin, vmax)

In [ ]:
cocitations = [1] * n_params + [0] * 2
bibcoupling = param_range + [1, 0]
slice_params = list(zip(cocitations, bibcoupling))

In [ ]:
fig, ((ax1, ax2, ax3), (ax4, ax5, ax6), (ax7, ax8, ax9), (ax10, ax11, ax12)) = plt.subplots(4, 3, figsize=(24, 32))

for (cc, bc), ax in zip(slice_params, 
                        [ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8, ax9, ax10, ax11, ax12]):
    df_part = mean_score_df[np.logical_and(mean_score_df.cocitation == cc, 
                                           mean_score_df.bibcoupling == bc)]
    
    param_grid_array = np.zeros((n_params, n_params))  
    for index, row in df_part.iterrows():
        citation = row.citation
        text_citation = row.text_citation
        param_grid_array[param_idx[citation], param_idx[text_citation]] = row[TARGET_COLUMN]
        
    plot_param_grid(param_grid_array, ax, 
                    vmin, vmax,
                    param_range, 
                    'Citation', 'Text Citation',
                    f'Cocitation: {cc}, Bibliographic Coupling: {bc}')

### Scores for Different Cluster Sizes

In [ ]:
# Very long!
score_by_size_df = get_score_by_size_dataframe_v1(results)

In [ ]:
score_by_size_df.head(10)

In [ ]:
def get_top_parameter_sets_per_size(score_df, target_col, n=5):
    return score_df.groupby(['Cluster Size'] + PARAM_NAMES)[target_col].mean().reset_index()

In [ ]:
def get_top_mean_score_per_size(score_df, target_col, size):
    top_scores_df = score_df.groupby(['Cluster Size'] + PARAM_NAMES)[target_col].mean().reset_index()
    top_scores_per_size = top_scores_df[top_scores_df['Cluster Size'] == size].sort_values(target_col, 
                                                                                           ascending=False)
    return top_scores_per_size[target_col].values[0]

In [ ]:
sizes = score_by_size_df['Cluster Size'].unique()

In [ ]:
for col in score_cols:
    print('\n', col, '\n')
    top_parameter_sets = get_top_parameter_sets_per_size(score_by_size_df, col)
    for size in sizes:
        print(f'{size}: {get_top_mean_score_per_size(score_by_size_df, col, size)}')
        print(top_parameter_sets[top_parameter_sets['Cluster Size'] == size][PARAM_NAMES + [col]].sort_values(col, 
                                                                                         ascending=False).head(3))

In [ ]:
fig, ax = plt.subplots()
for score_col in score_cols:
    mean_scores_per_size = [get_top_mean_score_per_size(score_by_size_df, score_col, size) for size in sizes]
    ax.plot(sizes, mean_scores_per_size, 'o-', label=score_col)
ax.set_xlabel('Cluster Size')
ax.set_ylabel('Score')
ax.legend()

## Comparison of Different Grid Search Results

In [ ]:
RESULT_FILES = {
    RESULTS_JSON_ORIGINAL: 1,
    RESULTS_JSON_COCIT_THRESHOLD: 2,
    RESULTS_JSON_NEW_EXPAND: 2
}

In [ ]:
DF_LABELS = ['original', 'cocitation_threshold', 'cocit_threshold\n+updated_expand']

In [ ]:
score_dfs = []

In [ ]:
for filename, version in RESULT_FILES.items():
    with open(filename, 'r') as f:
        results = json.load(f)
    if version == 1:
        score_df = get_score_dataframe_v1(results)
    elif version == 2:
        score_df = get_score_dataframe_v2(results)
    results = []
    score_dfs.append(score_df)

### Mean Score Comparison

In [ ]:
fig, ax = plt.subplots()
for score_col in score_cols:
    mean_scores_per_df = [get_top_mean_score(score_df, score_col) for score_df in score_dfs]
    ax.plot(DF_LABELS, mean_scores_per_df, 'o-', label=score_col)
ax.set_xlabel('Setting')
ax.set_ylabel('Score')
ax.legend()

### Top Parameters for Different Settings

In [ ]:
for col in score_cols:
    print('\n', col, '\n')
    for score_df, df_label in zip(score_dfs, DF_LABELS):
        print(f'{df_label}: {get_top_mean_score(score_df, col)}')
        print(get_top_parameter_sets(score_df, col, n=3), '\n')

### Best Clustering Scores Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for score_col, ax in zip(score_cols, axes):
    best_clustering_scores = []
    best_mean_scores = []
    for score_df, df_label in zip(score_dfs, DF_LABELS):
        best_clustering_score_df = get_best_clustering(score_df, score_col, unique=True)
        best_clustering_score_df['Level'] = best_clustering_score_df['Level'].apply(int)
        best_clustering_score_df['Setting'] = df_label
        best_clustering_scores.append(best_clustering_score_df)
        best_mean_scores.append(get_top_mean_score(score_df, score_col))
    best_clustering_scores_df = pd.concat(best_clustering_scores)
    sns.boxplot(ax=ax, x="Level", y=score_col, hue='Setting', data=best_clustering_scores_df)
    ax.step(str_levels, best_mean_scores, linestyle='--', c='red', where='mid', label='Mean Score')
    ax.legend()

## Generate Dummy Results to Test Visualizations

In [ ]:
import random
import itertools

from utils.io import get_review_pmids

random.seed(20210302)

In [ ]:
PARAM_RANGE = [0, 1, 2]
PARAM_NAMES = ['SIMILARITY_COCITATION',
               'SIMILARITY_BIBLIOGRAPHIC_COUPLING',
               'SIMILARITY_CITATION',
               'SIMILARITY_TEXT_CITATION']
PARAM_GRID = {p: PARAM_RANGE for p in PARAM_NAMES}

In [ ]:
def generate_grid_results():
    grid_results = []
    min_value = random.uniform(0.1, 0.3)
    max_value = min_value + random.uniform(0.2, 0.4)
    for param_values in itertools.product(*PARAM_GRID.values()):
        if sum(param_values) == 0:
            continue
        grid_results.append({
            'soi': list(param_values),
            'results': {
                'v_measure_score': random.uniform(min_value, max_value)
            }
        })
    return grid_results

def generate_dummy_results(n_reviews):
    results = []
    for pmid in get_review_pmids()[:n_reviews]:
        results.append({
            'pmid': pmid,
            'level': 1,
            'results': generate_grid_results()
        })
    return results

In [ ]:
results = generate_dummy_results(40)